# Learned Stage-Wise Curvature Tuning on Imagenette

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the Imagenette dataset using an ImageNet-pretrained ResNet-18.

Imagenette is a smaller subset of ImageNet containing ten visually distinct classes. Because the backbone is pretrained on ImageNet, this experiment helps evaluate how Baseline, S-CT, and Learned SW-CT behave when the downstream dataset is closely related to the original pretraining domain.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to examine whether a larger training budget improves performance or changes the learned stage-wise beta values.

Wall-clock execution time is also recorded for each method to compare computational cost.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [ ]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

In [1]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Imagenette Dataset

Before running the experiments, the Imagenette data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_imagenette",
    train_batch_size=32,
    test_batch_size=800,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

README.md: 0.00B [00:00, ?B/s]

imagenette.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Train samples: 7576
Validation samples: 1893
Test samples: 3925
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([6, 7, 2, 1, 1, 0, 6, 4, 4, 7])


# ResNet-18 Experiments

ResNet-18 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, Imagenette dataset split, experimental seeds, and downstream training protocol.

## ResNet-18 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-18 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [3]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 02:32:07.215 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_imagenette_resnet18_seed42_epochs20.log
2026-10-01 02:32:07.217 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 02:33:20.521 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 02:33:20.539 | INFO     | __main__:main:116 - Number of trainable parameters: 5130
2026-10-01 02:33:20.540 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 02:34:08.674 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.808238, Accuracy: 0.00%
2026-10-01 02:34:09.107 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.877600, Accuracy: 42.91%
2026-10-01 02:34:09.619 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.169286, Accuracy: 68.78%
2026-10-01 02:34:10.118 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.189595, Val Accuracy: 96.94%
2026-10-01 02:34:10.120 | INFO     | train:linear_probe:190 - New best validation acc

In [4]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 03:15:00.883 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_imagenette_resnet18_seed43_epochs20.log
2026-10-01 03:15:00.886 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 03:16:21.982 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 03:16:22.009 | INFO     | __main__:main:116 - Number of trainable parameters: 5130
2026-10-01 03:16:22.009 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 03:17:15.372 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.435476, Accuracy: 6.25%
2026-10-01 03:17:15.842 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.936718, Accuracy: 40.44%
2026-10-01 03:17:16.350 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.189015, Accuracy: 68.05%
2026-10-01 03:17:16.860 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.191367, Val Accuracy: 97.20%
2026-10-01 03:17:16.862 | INFO     | train:linear_probe:190 - New best validation acc

In [5]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 03:59:13.543 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_imagenette_resnet18_seed44_epochs20.log
2026-10-01 03:59:13.546 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 04:00:26.444 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 04:00:26.470 | INFO     | __main__:main:116 - Number of trainable parameters: 5130
2026-10-01 04:00:26.470 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 04:01:12.407 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.783121, Accuracy: 3.12%
2026-10-01 04:01:12.826 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.857830, Accuracy: 45.51%
2026-10-01 04:01:13.296 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.144844, Accuracy: 70.72%
2026-10-01 04:01:13.770 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.184070, Val Accuracy: 97.52%
2026-10-01 04:01:13.771 | INFO     | train:linear_probe:190 - New best validation acc

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [6]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 04:41:50.735 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet18_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 04:41:50.799 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 04:43:10.004 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 04:43:10.005 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 04:43:10.045 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 04:43:10.045 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 04:43:10.046 | INFO     | __main__:main:439 - Total trainable parameters: 5134
2026-10-01 04:43:11.631 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.588889, Accuracy: 0.00%
2026-10-01 04:43:30.258 | INFO     | train:train_epoch:47 

In [7]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 05:02:28.236 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet18_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 05:02:28.300 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 05:03:43.172 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 05:03:43.173 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 05:03:43.214 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 05:03:43.215 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 05:03:43.216 | INFO     | __main__:main:439 - Total trainable parameters: 5134
2026-10-01 05:03:44.630 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.639277, Accuracy: 15.62%
2026-10-01 05:04:03.570 | INFO     | train:train_epoch:47

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 05:22:14.434 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet18_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 05:22:14.497 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 05:23:03.394 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 05:23:03.395 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 05:23:03.421 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 05:23:03.422 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 05:23:03.422 | INFO     | __main__:main:439 - Total trainable parameters: 5134
2026-10-01 05:23:04.500 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.343274, Accuracy: 9.38%
2026-10-01 05:23:20.621 | INFO     | train:train_epoch:47 

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 05:39:00.439 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet18_seed42_epochs20_initrandom_betalr0.01.log
2026-10-01 05:39:00.506 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 05:39:49.324 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 05:39:49.324 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 05:39:49.351 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-01 05:39:49.352 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 05:39:49.352 | INFO     | __main__:main:439 - Total trainable parameters: 5134
2026-10-01 05:39:50.425 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.582472, Accuracy: 0.00%
2026-10-01 05:40:06.215 | INFO     | train:tra

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 06:18:45.568 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet18_seed43_epochs20_initrandom_betalr0.01.log
2026-10-01 06:18:45.656 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 06:19:55.767 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 06:19:55.767 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 06:19:55.805 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-01 06:19:55.806 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 06:19:55.806 | INFO     | __main__:main:439 - Total trainable parameters: 5134
2026-10-01 06:19:57.081 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.604656, Accuracy: 9.38%
2026-10-01 06:20:15.018 | INFO     | train:t

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 06:40:12.717 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.222556, Accuracy: 67.34%
2026-10-01 06:40:30.751 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.205813, Val Accuracy: 96.72%
2026-10-01 06:40:30.753 | INFO     | __main__:transfer:129 - Epoch 1: val_acc=96.72, betas=[0.8448, 0.8742, 0.8852, 0.9162]
2026-10-01 06:40:30.779 | INFO     | __main__:transfer:139 - New best validation accuracy: 96.72% at epoch 1
2026-10-01 06:40:31.329 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 0.328619, Accuracy: 90.62%
2026-10-01 06:40:49.965 | INFO     | train:train_epoch:47 - Epoch 2, Step 100, Loss: 0.193397, Accuracy: 96.57%
2026-10-01 06:41:08.611 | INFO     | train:train_epoch:47 - Epoch 2, Step 200, Loss: 0.165387, Accuracy: 96.70%
2026-10-01 06:41:26.609 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 0.102223, Val Accuracy: 97.52%
2026-10-01 06:41:26.610 | INFO     | __main__:transfer:129 - Epoch 2: val_acc=97.52, betas=[0.8733, 0.9

# ResNet-18 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 06:58:37.177 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_imagenette_resnet18_seed42_epochs30.log
2026-10-01 06:58:37.180 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 06:59:51.234 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 06:59:51.259 | INFO     | __main__:main:116 - Number of trainable parameters: 5130
2026-10-01 06:59:51.259 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 07:00:42.361 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.808238, Accuracy: 0.00%
2026-10-01 07:00:42.735 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.877600, Accuracy: 42.91%
2026-10-01 07:00:43.092 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.169286, Accuracy: 68.78%
2026-10-01 07:00:43.467 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.189595, Val Accuracy: 96.94%
2026-10-01 07:00:43.468 | INFO     | train:linear_probe:190 - New best validation acc

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 07:48:07.162 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_imagenette_resnet18_seed43_epochs30.log
2026-10-01 07:48:07.164 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 07:48:55.404 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 07:48:55.421 | INFO     | __main__:main:116 - Number of trainable parameters: 5130
2026-10-01 07:48:55.421 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 07:49:33.845 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.435476, Accuracy: 6.25%
2026-10-01 07:49:34.396 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.936718, Accuracy: 40.44%
2026-10-01 07:49:34.772 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.189015, Accuracy: 68.05%
2026-10-01 07:49:35.108 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.191367, Val Accuracy: 97.20%
2026-10-01 07:49:35.110 | INFO     | train:linear_probe:190 - New best validation acc

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 08:30:58.872 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_imagenette_resnet18_seed44_epochs30.log
2026-10-01 08:30:58.875 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 08:32:10.180 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 08:32:10.203 | INFO     | __main__:main:116 - Number of trainable parameters: 5130
2026-10-01 08:32:10.203 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 08:32:57.324 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.783121, Accuracy: 3.12%
2026-10-01 08:32:57.734 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.857830, Accuracy: 45.51%
2026-10-01 08:32:58.133 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.144844, Accuracy: 70.72%
2026-10-01 08:32:58.490 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.184070, Val Accuracy: 97.52%
2026-10-01 08:32:58.491 | INFO     | train:linear_probe:190 - New best validation acc

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 09:17:26.857 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet18_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 09:17:26.932 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 09:18:35.291 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 09:18:35.292 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 09:18:35.326 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 09:18:35.326 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 09:18:35.327 | INFO     | __main__:main:439 - Total trainable parameters: 5134
2026-10-01 09:18:36.722 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.588889, Accuracy: 0.00%
2026-10-01 09:18:54.536 | INFO     | train:train_epoch:47 

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 09:45:02.401 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet18_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 09:45:02.467 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 09:46:12.402 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 09:46:12.402 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 09:46:12.439 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 09:46:12.439 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 09:46:12.440 | INFO     | __main__:main:439 - Total trainable parameters: 5134
2026-10-01 09:46:13.749 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.639277, Accuracy: 15.62%
2026-10-01 09:46:31.714 | INFO     | train:train_epoch:47

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 10:12:47.460 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet18_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 10:12:47.539 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 10:13:56.693 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 10:13:56.693 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 10:13:56.722 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 10:13:56.722 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 10:13:56.722 | INFO     | __main__:main:439 - Total trainable parameters: 5134
2026-10-01 10:13:58.072 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.343274, Accuracy: 9.38%
2026-10-01 10:14:16.125 | INFO     | train:train_epoch:47 

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 10:41:25.094 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet18_seed42_epochs30_initrandom_betalr0.01.log
2026-10-01 10:41:25.159 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 10:42:25.195 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 10:42:25.196 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 10:42:25.226 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-01 10:42:25.226 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 10:42:25.227 | INFO     | __main__:main:439 - Total trainable parameters: 5134
2026-10-01 10:42:26.496 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.582472, Accuracy: 0.00%
2026-10-01 10:42:43.269 | INFO     | train:tra

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 11:09:51.644 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet18_seed43_epochs30_initrandom_betalr0.01.log
2026-10-01 11:09:51.749 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 11:11:05.541 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 11:11:05.542 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 11:11:05.588 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-01 11:11:05.589 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 11:11:05.589 | INFO     | __main__:main:439 - Total trainable parameters: 5134
2026-10-01 11:11:06.979 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.604656, Accuracy: 9.38%
2026-10-01 11:11:25.274 | INFO     | train:t

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 11:38:36.190 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_imagenette_resnet18_seed44_epochs30_initrandom_betalr0.01.log
2026-10-01 11:38:36.296 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 11:39:52.963 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 11:39:52.963 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 11:39:53.000 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-01 11:39:53.001 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 11:39:53.001 | INFO     | __main__:main:439 - Total trainable parameters: 5134
2026-10-01 11:39:54.491 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.329365, Accuracy: 6.25%
2026-10-01 11:40:12.357 | INFO     | train:t